# 📗 Bài 2 — Embedding & Tìm kiếm ngữ nghĩa

**Tuần 14**

> Đây là **nền tảng trực tiếp của RAG** (Phase 08). Mọi thứ trong bài này sẽ được dùng lại nguyên vẹn.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]
MAU_CHINH, MAU_NHAN = MAU[0], MAU[3]
torch.manual_seed(42)

DATA = Path("../../../data")
if not (DATA / "cau_mau.json").exists():
    DATA = Path("data")

cau_mau = json.loads((DATA / "cau_mau.json").read_text(encoding="utf-8"))
chu_de = sorted({c["chu_de"] for c in cau_mau})
print(f"{len(cau_mau)} câu, {len(chu_de)} chủ đề: {chu_de}")
print()
for c in cau_mau[:3]:
    print(f"  [{c['chu_de']:<10}] {c['cau']}")

## 1. Token ID là số TUỲ TIỆN

In [ ]:
print("  'vua'       -> token id 4821")
print("  'hoàng hậu' -> token id 9033")
print("  'bánh mì'   -> token id 1502")
print()
print("  Hai số 4821 và 9033 KHÔNG nói lên quan hệ gì.")
print("  4821 < 9033 không có nghĩa 'vua' nhỏ hơn 'hoàng hậu'.")
print()
print("  -> Cần EMBEDDING: biến mỗi token thành một VECTOR mang ngữ nghĩa.")

## 2. `nn.Embedding` chỉ là một bảng tra cứu

In [ ]:
emb = nn.Embedding(num_embeddings=10, embedding_dim=4)

print(f"Ma trận trọng số shape: {tuple(emb.weight.shape)}   (10 token × 4 chiều)")
print()
print("Tra token id 3:")
print(f"  emb(tensor([3])) = {emb(torch.tensor([3])).detach().numpy().round(3)}")
print(f"  weight[3]        = {emb.weight[3].detach().numpy().round(3)}")
print()
print("  -> Giống hệt nhau. Không có phép nhân nào, chỉ là LẤY HÀNG.")

**Tương đương với one-hot rồi nhân ma trận:** `one_hot(3) @ W == W[3]`. Nhưng `nn.Embedding` làm trực tiếp — nhanh hơn nhiều và ít bộ nhớ hơn. Với từ vựng 50.000, one-hot nghĩa là một vector 50.000 chiều toàn số 0 trừ một số 1.

**Vì sao các hàng lại "có nghĩa"?** Ban đầu chúng **ngẫu nhiên hoàn toàn**. Trong quá trình train, gradient đẩy các token xuất hiện trong ngữ cảnh giống nhau lại gần nhau. Sau khi train, `"vua"` và `"hoàng hậu"` ở gần nhau không phải vì ai đó lập trình thế, mà vì chúng hay xuất hiện ở những chỗ giống nhau.

> *"You shall know a word by the company it keeps"* — Firth, 1957.

In [ ]:
print(f"  {'model':<16} {'từ vựng':>10} {'chiều':>7} {'tham số embedding':>20}")
print("  " + "-" * 58)
for ten, v, d in [("GPT-2 nhỏ", 50_257, 768), ("GPT-2 lớn", 50_257, 1600),
                  ("Llama 3", 128_256, 4096)]:
    print(f"  {ten:<16} {v:>10,} {d:>7,} {v*d:>20,}")
print()
print("  Ma trận embedding thường là một trong những phần NẶNG nhất của model.")
print("  Đó là một lý do nữa khiến từ vựng không thể phình to tuỳ ý.")

## 3. ⭐ Cosine similarity

In [ ]:
def cosine(a, b):
    na, nb = torch.linalg.norm(a), torch.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(a @ b / (na * nb))


cap = [
    ("cùng hướng      ", [1., 0.], [1., 0.]),
    ("gần cùng hướng  ", [1., 0.], [0.9, 0.4]),
    ("vuông góc       ", [1., 0.], [0., 1.]),
    ("ngược hướng     ", [1., 0.], [-1., 0.]),
    ("cùng hướng, dài ", [1., 0.], [100., 0.]),
]

for mo_ta, a, b in cap:
    x, y = torch.tensor(a), torch.tensor(b)
    print(f"  {mo_ta} {str(a):<12} vs {str(b):<14} "
          f"tích vô hướng {float(x@y):>7.1f}   cosine {cosine(x, y):>6.3f}")

> 🔑 Nhìn dòng cuối: tích vô hướng nhảy lên **100**, nhưng cosine vẫn là **1.0**.
>
> **Vì sao điều này quan trọng với RAG:** đoạn văn **dài** có vector "lớn" hơn đoạn **ngắn**. Nếu dùng tích vô hướng thô, đoạn dài luôn thắng bất kể nội dung. Cosine chuẩn hoá điều đó, cho phép so sánh công bằng giữa đoạn 20 từ và đoạn 200 từ.

## 4. Mẹo quan trọng nhất: chuẩn hoá trước

In [ ]:
def chuan_hoa(X):
    d = torch.linalg.norm(X, dim=1, keepdim=True)
    return torch.where(d == 0, X, X / d.clamp(min=1e-12))


torch.manual_seed(0)
kho = torch.randn(1000, 128)
truy_van = torch.randn(128)

import time

t = time.perf_counter()
diem_cham = torch.tensor([cosine(truy_van, v) for v in kho])
t_cham = time.perf_counter() - t

t = time.perf_counter()
kho_n = chuan_hoa(kho)
tv_n = truy_van / torch.linalg.norm(truy_van)
diem_nhanh = kho_n @ tv_n
t_nhanh = time.perf_counter() - t

print(f"  Vòng lặp        : {t_cham*1000:8.2f} ms")
print(f"  Nhân ma trận    : {t_nhanh*1000:8.2f} ms")
print(f"  Nhanh hơn khoảng {t_cham/max(t_nhanh,1e-9):.0f} lần")
print(f"  Kết quả giống nhau? {torch.allclose(diem_cham, diem_nhanh, atol=1e-5)}")

Sau khi chuẩn hoá (`|a| = |b| = 1`):
```
cos(a, b) = a · b / (1 × 1) = a · b
```

**Cosine trở thành tích vô hướng.** Thay vì tính cosine cho từng cặp, ta làm **một** phép nhân ma trận cho toàn bộ kho.

> 📌 **Mọi vector database** (Chroma, Qdrant, Pinecone) đều lưu vector **đã chuẩn hoá sẵn** vì lý do này.

## 5. ⭐⭐ Mô phỏng tìm kiếm ngữ nghĩa

Ta chưa có model embedding thật (Phase 08 sẽ dùng). Ở đây ta **mô phỏng** bằng cách tạo vector sao cho câu cùng chủ đề nằm gần nhau — để thấy rõ *cơ chế* hoạt động.

In [ ]:
torch.manual_seed(0)
D_EMB = 32

# Mỗi chủ đề có một "tâm" riêng; câu = tâm + nhiễu
tam = {cd: torch.randn(D_EMB) for cd in chu_de}
vector_cau = torch.stack([
    tam[c["chu_de"]] + torch.randn(D_EMB) * 0.45 for c in cau_mau
])
vector_cau = chuan_hoa(vector_cau)

print(f"Kho vector: {tuple(vector_cau.shape)}")


def tim_kiem(vec_truy_van, k=5):
    diem = vector_cau @ (vec_truy_van / torch.linalg.norm(vec_truy_van))
    diem_top, idx = torch.topk(diem, k)
    return [(cau_mau[i]["cau"], cau_mau[i]["chu_de"], float(d))
            for i, d in zip(idx, diem_top)]


# "Câu hỏi" về ẩm thực
cau_hoi = tam["am_thuc"] + torch.randn(D_EMB) * 0.3
print("\nCâu hỏi thuộc chủ đề 'am_thuc'. Top 5 kết quả:\n")
for cau, cd, diem in tim_kiem(cau_hoi):
    print(f"  {diem:.3f}  [{cd:<10}] {cau}")

In [ ]:
# Kiểm tra: tìm kiếm có trả về ĐÚNG chủ đề không?
print(f"  {'chủ đề truy vấn':<14} {'top-5 đúng chủ đề':>20}")
print("  " + "-" * 36)
for cd in chu_de:
    dung = 0
    for _ in range(20):
        q = tam[cd] + torch.randn(D_EMB) * 0.3
        dung += sum(1 for _, c, _ in tim_kiem(q, 5) if c == cd)
    print(f"  {cd:<14} {dung/20:>18.1f}/5")

> 🔗 **Đây chính xác là cơ chế của mọi hệ thống RAG:**
> ```
> ① Biến câu hỏi thành vector
> ② Biến mọi đoạn tài liệu thành vector
> ③ Tìm đoạn có cosine cao nhất
> ④ Nhét đoạn đó vào prompt
> ```
> Không có gì hơn thế. Phase 08 chỉ thay bước ① và ② bằng model embedding thật.

## 6. Ma trận tương đồng — nhìn thấy cấu trúc

In [ ]:
# Sắp xếp câu theo chủ đề để cấu trúc lộ rõ
thu_tu = sorted(range(len(cau_mau)), key=lambda i: cau_mau[i]["chu_de"])
V = vector_cau[thu_tu]
M = (V @ V.T).numpy()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1)

ranh = []
truoc = None
for vi_tri, i in enumerate(thu_tu):
    cd = cau_mau[i]["chu_de"]
    if cd != truoc:
        ranh.append((vi_tri, cd)); truoc = cd
for vi_tri, _ in ranh[1:]:
    ax.axhline(vi_tri - .5, color="black", lw=.8)
    ax.axvline(vi_tri - .5, color="black", lw=.8)

ax.set_xticks([v for v, _ in ranh], [c for _, c in ranh], rotation=30, ha="right")
ax.set_yticks([v for v, _ in ranh], [c for _, c in ranh])
ax.grid(visible=False)
plt.colorbar(im, ax=ax, shrink=.8, label="cosine similarity")
plt.title("Ma trận tương đồng — sắp theo chủ đề")
plt.tight_layout(); plt.show()

Các **khối vuông đỏ** dọc đường chéo chính là các chủ đề: câu trong cùng chủ đề có cosine cao với nhau.

**Ba kiểm tra nhanh khi debug embedding:**
1. Đường chéo chính phải bằng `1.0`
2. Ma trận phải **đối xứng**
3. Mọi giá trị phải trong `[-1, 1]`

Nếu một trong ba điều trên sai → bạn có bug.

**Ứng dụng thực tế của ma trận này:** tìm tài liệu trùng lặp (cosine > 0.95) · gom cụm văn bản · phát hiện đạo văn · **đánh giá chất lượng embedding** (các câu cùng chủ đề có thật sự gần nhau không?).

## 7. Embedding cho CÂU — và cái bẫy padding

In [ ]:
emb = nn.Embedding(100, 8)

# Batch hai câu, độ dài khác nhau -> phải đệm
ids = torch.tensor([[12, 45, 78, 0, 0],
                    [33, 21, 56, 90, 11]])
mask = torch.tensor([[1, 1, 1, 0, 0],
                     [1, 1, 1, 1, 1]])

# ❌ SAI: tính trung bình kể cả phần đệm
sai = emb(ids).mean(dim=1)

# ✅ ĐÚNG: bỏ qua padding
vec = emb(ids)
m = mask.unsqueeze(-1).float()
dung = (vec * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)

that_su = emb(torch.tensor([12, 45, 78])).mean(dim=0)

print(f"  Câu 1 — cách SAI  : {sai[0].detach().numpy().round(3)}")
print(f"  Câu 1 — cách ĐÚNG : {dung[0].detach().numpy().round(3)}")
print(f"  Câu 1 — thật sự   : {that_su.detach().numpy().round(3)}")
print()
print(f"  Sai lệch của cách SAI: {float((sai[0]-that_su).abs().mean()):.4f}")
print(f"  Sai lệch của cách ĐÚNG: {float((dung[0]-that_su).abs().mean()):.6f}")

> ⚠️ **Lỗi này rất hay gặp và rất khó phát hiện.** Model vẫn chạy, kết quả vẫn "có vẻ" hợp lý, chỉ là chất lượng kém hơn đáng kể.
>
> **Dấu hiệu nhận biết:** nếu các câu **ngắn** bị tìm kiếm sai nhiều hơn câu dài — hãy kiểm tra mask.

**Điểm yếu của mean pooling:** nó **bỏ qua thứ tự từ**.

In [ ]:
a = emb(torch.tensor([12, 45, 78])).mean(dim=0)
b = emb(torch.tensor([78, 45, 12])).mean(dim=0)

print(f"  'mèo đuổi chuột' vs 'chuột đuổi mèo'")
print(f"  Vector giống hệt nhau? {torch.allclose(a, b)}")
print()
print("  -> Trung bình cộng không quan tâm thứ tự.")
print("  -> Đó chính là vấn đề mà TRANSFORMER giải quyết (bài 3).")

## 8. Tổng kết

| Khái niệm | Ý nghĩa |
|---|---|
| **Embedding** | Bảng tra cứu: token id → vector mang ngữ nghĩa |
| **Cosine similarity** | Đo độ giống nhau theo *hướng*, không theo độ lớn |
| **Chuẩn hoá trước** | Biến cosine thành một phép nhân ma trận |
| **Mean pooling** | Cách đơn giản nhất tạo vector cho câu |
| **Mask** | Bắt buộc khi có padding |

**Điểm yếu cần nhớ:** token embedding cho `"đá"` trong *"nước đá"* và *"đá bóng"* là **cùng một vector** — sai. Chuyển từ embedding tĩnh sang **contextual embedding** chính là công việc của **attention**.

## ✍️ Bài tập

```bash
pytest tests/phase06/test_ex02.py -v
```